# Projet CoCoMa - coordination de satellites
---

Victor Fleiser - Thomas Marchand

In [ ]:
# ------------------------------ IMPORTS ------------------------------

import random
import plotly.graph_objects as go
import subprocess
import json
import tempfile
import time

# You may need to upgrade the 'nbformat' package for the plot to work in Jupyter notebooks.
# python -m pip install --upgrade nbformat

In [ ]:
# ------------------------------ HELPER FUNCTIONS ------------------------------
def overlap(time_start1: int, time_end1: int, time_start2: int, time_end2: int) -> bool:
	# returns True if the time intervals (time_start1, time_end1) and (time_start2, time_end2) overlap
	# this assumes time_start < time_end for both intervals
	return max(time_start1, time_start2) < min(time_end1, time_end2)

def is_within(time_start_inner: int, time_end_inner: int, time_start_outer: int, time_end_outer: int) -> bool:
	# returns True if the time interval (time_start_inner, time_end_inner) is within (time_start_outer, time_end_outer)
	return time_start_outer <= time_start_inner and time_end_inner <= time_end_outer


In [ ]:
# ------------------------------ DATA STRUCTURES ------------------------------
class Satellite:
	def __init__(self, name: str, time_start: int, time_end: int, capacity: int, transition_time: int):
		self.name: str = name
		self.time_start: int = time_start	# Satellite availability start time
		self.time_end: int = time_end	# Satellite availability end time
		self.capacity: int = capacity	# Maximum number of observations the satellite can perform per period
		self.transition_time: int = transition_time	# Time required for the satellite to transition between 2 observations
		self.exclusive_windows: list = []	# List of ExclusiveWindow objects
		# indicates the time periods during which specific users have exclusive access to this satellite
		
		# the following paramater is used when the satellite has at most 1 exclusive user (not a thing in the paper, but present in the subject)
		self.exclusive_user: User = None

	def add_exclusive_time(self, user: 'User', time_start: int, time_end: int):
		# Adds an exclusive time period for a user if no conflicts exist

		# Check for conflicts with existing exclusive times
		for window in self.exclusive_windows:
			if overlap(time_start, time_end, window.time_start, window.time_end):
				return False	# Conflict detected, cannot add exclusive time
		self.exclusive_windows.append(ExclusiveWindow(user, self, time_start, time_end))
		return True	# Exclusive time added successfully

class User:
	def __init__(self, name: str, exclusive_windows: list, priority: int):
		self.name: str = name
		self.exclusive_windows: list = exclusive_windows	# List of ExclusiveWindow objects
		# indicates the time periods during which the user has exclusive access to specific satellites
		self.priority: int = priority	# Priority level of the user, lower number indicates higher priority

	def get_id(self) -> int:
		# Extracts and returns the numerical ID of the user from its name
		return int(self.name.split('_')[1])

class ExclusiveWindow:
	def __init__(self, user: User, satellite: Satellite, time_start: int, time_end: int):
		self.name: str = f"e_{user.get_id()}_{satellite.name}_{time_start}_{time_end}"
		self.user: User = user
		self.satellite: Satellite = satellite
		self.time_start: int = time_start
		self.time_end: int = time_end
	
	def can_potentially_assign_observation(self, observation: 'Observation') -> bool:
		# checks if the exclusive window can potentially assign the given observation (only checks for overlap, not for the observation's length)
		if self.satellite.name != observation.satellite.name:
			return False
		return overlap(observation.time_start, observation.time_end, self.time_start, self.time_end)

class Request:
	def __init__(self, name: str, time_start: int, time_end: int, duration: int, reward: int, position, user: User, observations: list):
		self.name: str = name
		self.time_start: int = time_start	# Request availability start time
		self.time_end: int = time_end	# Request availability end time
		self.duration: int = duration	# Duration required to fulfill the request
		self.reward: int = reward	# Reward associated with fulfilling the request
		self.position = position	# Position required for the observations : LLA coordinates (TODO: define this)
		self.user: User = user	# User who made the request
		self.observations: list = observations	# List of possible observations for this request

	def get_id(self) -> int:
		# Extracts and returns the numerical ID of the request from its name
		return int(self.name.split('_')[2])

class Observation:
	def __init__(self, name: str, time_start: int, time_end: int, duration: int, request: Request, reward: int, satellite: Satellite):
		self.name: str = name
		self.time_start: int = time_start	# Observation start time
		self.time_end: int = time_end	# Observation end time
		self.duration: int = duration	# Observation duration
		self.request: Request = request	# Associated request
		self.reward: int = reward	# Observation reward
		self.satellite: Satellite = satellite	# Satellite performing the observation
		self.user: User = request.user	# User associated with the observation
		self.priority: int = request.user.priority	# Priority of the user associated with the observation

	def get_id(self) -> int:
		# Extracts and returns the numerical ID of the observation from its name
		return int(self.name.split('_')[3])

class Problem:
	def __init__(self, satellites: dict, users: dict, requests: dict, observations: dict):
		self.satellites: dict = satellites
		self.users: dict = users
		self.requests: dict = requests
		self.observations: dict = observations

	def get_exclusive_users(self) -> dict:
		# returns a dict of exclusive users in the problem
		return {user.name: user for user in self.users.values() if user.name != "u_0"}
	
	def get_user_requests(self, user: User) -> dict:
		# returns a dict of requests made by the given user
		return {request.name: request for request in self.requests.values() if request.user.name == user.name}
	
	def get_user_observations(self, user: User) -> dict:
		# returns a dict of observations associated with the given user
		return {obs.name: obs for obs in self.observations.values() if obs.user.name == user.name}
	
	def get_observations_outside_exclusive_windows(self) -> dict:
		# returns a dict of observations that are outside exclusive windows, (all exclusive user observations are already guaranteed on exclusive windows)
		observations_outside_exclusive_windows = {}
		for obs in self.observations.values():
			if obs.user.name != "u_0":
				continue	# skip exclusive user observations since they are guaranteed on exclusive windows
			sat_exclusive_windows = obs.satellite.exclusive_windows
			is_outside = True
			for window in sat_exclusive_windows:
				if overlap(obs.time_start, obs.time_end, window.time_start, window.time_end):
					is_outside = False
					break
			if is_outside:
				observations_outside_exclusive_windows[obs.name] = obs
		return observations_outside_exclusive_windows

	def get_request_copy_without_observations(self, request: Request) -> Request:
		# returns a copy of the given request
		return Request(request.name, request.time_start, request.time_end, request.duration, request.reward, request.position, request.user, [])

	def get_non_exclusive_requests(self) -> dict:
		# returns a dict of non-exclusive requests in the problem
		return {request.name: request for request in self.requests.values() if request.user.name == "u_0"}

class Solution:
	def __init__(self, problem: Problem, scheduled_observations: list):
		self.satellites: dict = problem.satellites
		self.users: dict = problem.users
		self.requests: dict = problem.requests
		self.observations: dict = problem.observations
		self.scheduled_observations: list = scheduled_observations	# List of tuples (observation, scheduled_start_time)

	def verify(self) -> bool:
		success = True
		# verifies if the solution is valid according to the following constraints :
		# 1. Check that observation/request/satellite time windows are respected
		# 2. Check that satellite capacity is not exceeded
		# 3. Check that satellite observations do not collide
		# 4. Check that satellite transition times are respected
		# 5. Check that all exclusive user requests are fulfilled
		# 6. Check that each request is fulfilled at most once

		# 1. Check that observation/request/satellite time windows are respected
		for obs, start_time in self.scheduled_observations:
			end_time = start_time + obs.request.duration
			# 1.1 Check observation time window
			if not is_within(start_time, end_time, obs.time_start, obs.time_end):
				print(f"[verify C1.1] Observation {obs.name} scheduled time [{start_time}, {end_time}] is outside its time window [{obs.time_start}, {obs.time_end}].")
				success = False
			# 1.2 Check request time window
			if not is_within(start_time, end_time, obs.request.time_start, obs.request.time_end):
				print(f"[verify C1.2] Observation {obs.name} scheduled time [{start_time}, {end_time}] is outside its request {obs.request.name} time window [{obs.request.time_start}, {obs.request.time_end}].")
				success = False
			# 1.3 Check satellite availability window
			if not is_within(start_time, end_time, obs.satellite.time_start, obs.satellite.time_end):
				print(f"[verify C1.3] Observation {obs.name} scheduled time [{start_time}, {end_time}] is outside its satellite {obs.satellite.name} availability window [{obs.satellite.time_start}, {obs.satellite.time_end}].")
				success = False
		
		# 2. Check that satellite capacity is not exceeded
		satellite_observation_counts = {sat_name: 0 for sat_name in self.satellites.keys()}
		for obs, start_time in self.scheduled_observations:
			satellite_observation_counts[obs.satellite.name] += 1
		for sat_name, count in satellite_observation_counts.items():
			if count > self.satellites[sat_name].capacity:
				print(f"[verify C2] Satellite {sat_name} has {count} scheduled observations, exceeding its capacity of {self.satellites[sat_name].capacity}.")
				success = False

		# 3. Check that satellite observations do not collide
		satellite_schedules = {sat_name: [] for sat_name in self.satellites.keys()}
		for obs, start_time in self.scheduled_observations:
			end_time = start_time + obs.request.duration
			satellite_schedules[obs.satellite.name].append((start_time, end_time, obs.name))
		for sat_name, schedule in satellite_schedules.items():
			# sort schedule by start time
			schedule.sort(key=lambda x: x[0])
			for i in range(len(schedule) - 1):
				if overlap(schedule[i][0], schedule[i][1], schedule[i+1][0], schedule[i+1][1]):
					print(f"[verify C3] Satellite {sat_name} has overlapping observations {schedule[i][2]} and {schedule[i+1][2]} scheduled at [{schedule[i][0]}, {schedule[i][1]}] and [{schedule[i+1][0]}, {schedule[i+1][1]}].")
					success = False

		# 4. Check that satellite transition times are respected
		for sat_name, schedule in satellite_schedules.items():
			# schedule is already sorted by start time
			for i in range(len(schedule) - 1):
				transition_time = schedule[i+1][0] - schedule[i][1]
				required_transition_time = self.satellites[sat_name].transition_time
				if transition_time < required_transition_time:
					print(f"[verify C4] Satellite {sat_name} does not respect transition time between observations {schedule[i][2]} and {schedule[i+1][2]}. Required: {required_transition_time}, Actual: {transition_time}.")
					success = False

		# 5. Check that all exclusive user requests are fulfilled
		for request in self.requests.values():
			if request.user.name == "u_0":
				continue	# skip non-exclusive user requests
			fulfilled = any(obs.request.name == request.name for obs, start_time in self.scheduled_observations)
			if not fulfilled:
				print(f"[verify C5] Exclusive user request {request.name} by user {request.user.name} is not fulfilled in the solution.")
				success = False
		
		# 6. Check that each request is fulfilled at most once
		fulfilled_requests = {}
		for obs, start_time in self.scheduled_observations:
			if obs.request.name in fulfilled_requests:
				print(f"[verify C6] Request {obs.request.name} is fulfilled more than once in the solution (observations {fulfilled_requests[obs.request.name]} and {obs.name}).")
				success = False
			else:
				fulfilled_requests[obs.request.name] = obs.name

		return success

	def total_reward(self) -> int:
		# calculates the total reward of the scheduled observations in the solution
		return sum(obs.reward for obs, start_time in self.scheduled_observations)

	def get_solution_schedules(self) -> dict:
		# returns a dictionary mapping observation names to their scheduled start and end times (used for visualization)
		return {obs.name: (start_time, start_time + obs.request.duration) for obs, start_time in self.scheduled_observations}

	def get_completed_requests(self) -> list:
		# returns a list of request names that have been completed in the solution
		return list(set(obs.request.name for obs, start_time in self.scheduled_observations))
	
	def get_uncompleted_requests(self) -> list:
		# returns a list of request names that have not been completed in the solution
		completed_requests = set(self.get_completed_requests())
		return [request.name for request in self.requests.values() if request.name not in completed_requests]
	
	def get_satellite_schedule(self, satellite: Satellite) -> list:
		# returns the schedule of the given satellite as a list of tuples (start_time, end_time, observation_name)
		schedule = []
		for obs, start_time in self.scheduled_observations:
			if obs.satellite.name == satellite.name:
				schedule.append((start_time, start_time + obs.request.duration, obs.name))
		return schedule

	def get_proportion_of_completed_requests(self) -> float:
		# returns the proportion of completed requests in the solution, as well as the number of completed and total requests
		total_requests = len(self.requests)
		if total_requests == 0:
			return 0.0
		completed_requests = len(self.get_completed_requests())
		return completed_requests / total_requests, completed_requests, total_requests

In [ ]:
# ------------------------------ HELPER FUNCTIONS 2 ------------------------------
def check_overlap_between_exclusive_and_normal_times(satellite: Satellite, request_start: int, request_end: int) -> bool:
	# we don't allow observations overlapping both exclusive and non-exclusive times at the same time, or overlapping 2 different exclusive times
	# so a request must be fully within either an exclusive time or a non-exclusive time on the satellite
	# returns True if the request is valid (fully within either exclusive or non-exclusive time), False otherwise
	for window in satellite.exclusive_windows:
		if is_within(request_start, request_end, window.time_start, window.time_end):
			return True	# fully within an exclusive time
		if overlap(request_start, request_end, window.time_start, window.time_end):
			return False	# overlaps an exclusive time and is not fully within it
	# if we reach here, the request does not overlap any exclusive time
	return True	# fully within a non-exclusive time

def sort_and_extend_satellite_schedule(satellite: Satellite, schedule: list) -> list:
	# sorts and extends the satellite's schedule to include transition times between observations
	# schedule is a list of tuples (start_time, end_time, observation_name)
	transition_time = satellite.transition_time
	sorted_schedule = sorted(schedule, key=lambda x: x[0])	# sort by start_time
	updated_schedule = []
	for start_time, end_time, obs_name in sorted_schedule:
		updated_schedule.append((start_time - transition_time, end_time + transition_time, obs_name))

	return updated_schedule

def get_observations_in_time_slot(start_time: int, end_time: int, satellite_schedule: list) -> list[Observation]:
		# returns the list of observations scheduled in the given time slot
		observations_in_slot = []
		for scheduled_start, scheduled_end, obs_name in satellite_schedule:
			if overlap(start_time, end_time, scheduled_start, scheduled_end):
				observations_in_slot.append(obs_name)
		return observations_in_slot

def get_best_observation_replacement(observation: Observation, satellite_schedule: list, problem: Problem) -> tuple[int, int]:
	# tests all possible time slots of the observation on the satellite's schedule, returns the best max reward and corresponding start time
	observation_start_window = observation.time_start
	observation_end_window = observation.time_end
	observation_duration = observation.duration
	observation_reward = observation.reward
	max_reward = 0
	best_start_time = None
	for start_time in range(observation_start_window, observation_end_window - observation_duration + 1):
		# insert the observation at start_time
		end_time = start_time + observation_duration
		total_reward = observation_reward
		# reduce reward by each overlapping observation's reward that would be removed
		overlapping_observations = get_observations_in_time_slot(start_time, end_time, satellite_schedule)
		observations_reward = [problem.observations[obs_name].reward for obs_name in overlapping_observations]
		total_reward -= sum(observations_reward)
		if total_reward > max_reward:
			max_reward = total_reward
			best_start_time = start_time
	return max_reward, best_start_time

def get_exclusive_user_from_observation_assignment(observation: Observation) -> User:
	# returns the exclusive user that would execute the given observation
	for window in observation.satellite.exclusive_windows:
		if window.can_potentially_assign_observation(observation):
			return window.user
	return None

def remove_overlapping_observations(new_observation: Observation, new_start_time: int, user_solution: Solution) -> list[tuple[Observation, int]]:
	# removes any observation in the user_solution that overlaps with the new_observation at new_start_time
	new_end_time = new_start_time + new_observation.duration
	observations_to_remove = []
	for obs, start_time in user_solution.scheduled_observations:
		end_time = start_time + obs.duration
		if obs.satellite.name == new_observation.satellite.name:
			if overlap(new_start_time, new_end_time, start_time, end_time):
				observations_to_remove.append((obs, start_time))
	for obs in observations_to_remove:
		user_solution.scheduled_observations.remove(obs)
	return observations_to_remove

# Générateur aléatoire d'instances

In [ ]:
# ------------------------------ RANDOM INSTANCE GENERATION ------------------------------


class RandomInstanceGenerator:
	# all the parameters used to generate a random instance
	def __init__(self,
		computing_time: float = None,
		number_of_satellites: int = None,
		capacity_per_satellite: int = None,
		satellites_availability_window : tuple = None,
		transition_time_per_satellite: int = None,
		number_of_exclusive_users: int = None,
		exclusive_portions_per_user: int = None,
		exclusive_portion_duration_range: tuple = None,
		number_of_non_exclusive_requests_range: tuple = None,
		number_of_exclusive_requests_per_exclusive_user_range: tuple = None,
		request_duration: int = None,
		request_availability_window: tuple = None,
		reward_non_exclusive_request_range: tuple = None,
		reward_exclusive_request_range: list = None,
		number_of_observations_per_request: int = None,
		observation_time_window_range: tuple = None
		):

		self.computing_time = computing_time
		self.number_of_satellites = number_of_satellites
		self.capacity_per_satellite = capacity_per_satellite
		self.satellites_availability_window = satellites_availability_window
		self.transition_time_per_satellite = transition_time_per_satellite
		self.number_of_exclusive_users = number_of_exclusive_users
		self.exclusive_portions_per_user = exclusive_portions_per_user
		self.exclusive_portion_duration_range = exclusive_portion_duration_range
		self.number_of_non_exclusive_requests_range = number_of_non_exclusive_requests_range
		self.number_of_exclusive_requests_per_exclusive_user_range = number_of_exclusive_requests_per_exclusive_user_range
		self.request_duration = request_duration
		self.request_availability_window = request_availability_window
		self.reward_non_exclusive_request_range = reward_non_exclusive_request_range
		self.reward_exclusive_request_range = reward_exclusive_request_range
		self.number_of_observations_per_request = number_of_observations_per_request
		self.observation_time_window_range = observation_time_window_range

		self.exclusive_windows_generation_method = "subject"	# "paper" or "subject" version of exclusive times generation
		self.non_exclusive_observations_anywhere = False	# if True, non-exclusive observations can be anywhere (not just fully within exclusive or non-exclusive times)
		self.verbose_results = False	# if True, prints every satellite/user/request/observation generated

	def generate_satellites(self) -> dict:
		# Step 1. generating the satellites
		satellites = {}
		for i in range(self.number_of_satellites):
			satellite = Satellite(
				name = f"s_{i}",
				time_start = self.satellites_availability_window[0],
				time_end = self.satellites_availability_window[1],
				capacity = self.capacity_per_satellite,
				transition_time = self.transition_time_per_satellite
			)
			satellites[satellite.name] = satellite
		return satellites

	def generate_users(self) -> dict:
		# Step 2. generating the users
		# 2.1 generating the exclusive users
		users = {}
		for i in range(self.number_of_exclusive_users):
			user = User(
				name = f"u_{i+1}",
				exclusive_windows = [],	# to be filled in step 3.
				priority = i + 1	# priority based on user index (lower index = higher priority)
			)
			users[user.name] = user
		# 2.2 generating non-exclusive users (all aggregated into a single user)
		non_exclusive_user = User(
			name = "u_0",
			exclusive_windows = [],	# no exclusive windows for non-exclusive user
			priority = self.number_of_exclusive_users + 1	# priority after all exclusive users
		)
		users[non_exclusive_user.name] = non_exclusive_user
		return users

	def generate_exclusive_windows_paper_version(self, satellites: dict, users: dict):
		# Alternate version of step 3. that lets every exclusive user have exclusive times on every satellite (Paper). See new version below for the subject's version
		# 3. generating the user exclusive times
		for user in users.values():
			if user.name == "u_0":
				continue	# skip non-exclusive user
			for _ in range(self.exclusive_portions_per_user):
				attempts_allowed = 1000	# Limit the number of attempts to find a non-conflicting exclusive time to avoid infinite loops
				attempts = 0
				success = False
				while not success and attempts < attempts_allowed:
					attempts += 1
					# Randomly select a satellite for the exclusive time
					satellite = random.choice(list(satellites.values()))
					# Randomly generate a start time and duration for the exclusive time
					exclusive_duration = random.randint(self.exclusive_portion_duration_range[0], self.exclusive_portion_duration_range[1])
					exclusive_start = random.randint(satellite.time_start, satellite.time_end - exclusive_duration)
					exclusive_end = exclusive_start + exclusive_duration
					# Attempt to add the exclusive time to the satellite
					success = satellite.add_exclusive_time(user, exclusive_start, exclusive_end)
					if success:
						user.exclusive_windows.append((satellite, (exclusive_start, exclusive_end)))
				if not success:
					print(f"Warning: Could not assign exclusive time for user {user.name} after {attempts_allowed} random assignement attempts.")

	def generate_exclusive_windows_subject_version(self, satellites: dict, users: dict):
		# Step 3. generating the user exclusive times (subject version : each satellite has exactly 1 exclusive user)
		# 3.1 Assign each exclusive user to a satellite
		exclusive_users = [user for user in users.values() if user.name != "u_0"]
		random.shuffle(exclusive_users)

		if self.number_of_exclusive_users > self.number_of_satellites:
			# Too many exclusive users to assign at most one per satellite. THIS IS NOT VALID FOR THE SUBJECT
			raise SystemExit("Error: More exclusive users than satellites. Cannot assign unique exclusive users to satellites.")

		for i in range(self.number_of_exclusive_users):
			satellite = satellites[f"s_{i}"]
			user = exclusive_users[i]
			satellite.exclusive_user = user

		# 3.2 Assign the rest of the satellites :
		exclusive_users_and_empty = exclusive_users.copy() + [None]	# allow some satellites to have no exclusive user
		for i in range(self.number_of_exclusive_users, self.number_of_satellites):
			satellite = satellites[f"s_{i}"]
			user = random.choice(exclusive_users_and_empty)
			satellite.exclusive_user = user

		# 3.3 generating the user exclusive times
		for user in users.values():
			if user.name == "u_0":
				continue	# skip non-exclusive user
			# Get the list of satellites for which this user is the exclusive user
			user_exclusive_satellites = [sat for sat in satellites.values() if sat.exclusive_user == user]
			if not user_exclusive_satellites:
				print(f"Warning: User {user.name} has no exclusive satellites assigned, BUG?.")
				continue	# this user has no exclusive satellites assigned, should not happen
			for j in range(self.exclusive_portions_per_user):
				attempts_allowed = 1000	# Limit the number of attempts to find a non-conflicting exclusive time to avoid infinite loops
				attempts = 0
				success = False
				while not success and attempts < attempts_allowed:
					attempts += 1
					# Randomly select one of the user's exclusive satellites for the exclusive time
					satellite = random.choice(user_exclusive_satellites)
					# Randomly generate a start time and duration for the exclusive time
					exclusive_duration = random.randint(self.exclusive_portion_duration_range[0], self.exclusive_portion_duration_range[1])
					exclusive_start = random.randint(satellite.time_start, satellite.time_end - exclusive_duration)
					exclusive_end = exclusive_start + exclusive_duration
					# Attempt to add the exclusive time to the satellite
					success = satellite.add_exclusive_time(user, exclusive_start, exclusive_end)
					if success:
						user.exclusive_windows.append(ExclusiveWindow(user, satellite, exclusive_start, exclusive_end))
				if not success:
					print(f"Warning: Could not assign exclusive time for user {user.name} after {attempts_allowed} random assignement attempts.")

	def generate_requests(self, users: dict) -> dict:
		# 4. generating the requests for each user
		requests = {}
		# 4.1 generating non-exclusive requests
		number_of_non_exclusive_requests = random.randint(self.number_of_non_exclusive_requests_range[0], self.number_of_non_exclusive_requests_range[1])
		for i in range(number_of_non_exclusive_requests):
			request_start = self.request_availability_window[0]
			request_end = self.request_availability_window[1]
			request = Request(
				name = f"r_0_{i}",
				time_start = request_start,
				time_end = request_end,
				duration = self.request_duration,
				reward = random.randint(self.reward_non_exclusive_request_range[0], self.reward_non_exclusive_request_range[1]),
				position = None,	# position generation not implemented yet
				user = users["u_0"],
				observations = []	# to be filled in step 5.
			)
			requests[request.name] = request
		# 4.2 generating exclusive requests for each exclusive user
		for user in users.values():
			if user.name == "u_0":
				continue	# skip non-exclusive user
			number_of_exclusive_requests = random.randint(self.number_of_exclusive_requests_per_exclusive_user_range[0], self.number_of_exclusive_requests_per_exclusive_user_range[1])
			for i in range(number_of_exclusive_requests):
				request_start = self.request_availability_window[0]
				request_end = self.request_availability_window[1]
				request = Request(
					name = f"r_{user.get_id()}_{i}",
					time_start = request_start,
					time_end = request_end,
					duration = self.request_duration,
					reward = random.choice(self.reward_exclusive_request_range),
					position = None,	# position generation not implemented yet
					user = user,
					observations = []	# to be filled in step 5.
				)
				requests[request.name] = request

		return requests
	
	def generate_observations(self, satellites: dict, requests: dict) -> dict:
		# Step 5. generating the observations for each request
		observations = {}
		for request in requests.values():
			for obs_index in range(self.number_of_observations_per_request):
				attempts_allowed = 1000	# Limit the number of attempts to find valid observations to avoid infinite loops
				attempts = 0
				success = False
				while not success and attempts < attempts_allowed:
					attempts += 1
					if request.user.name == "u_0":
						# Non-exclusive user request

						# pick a random satellite
						satellite = random.choice(list(satellites.values()))
						# pick a random time window for the observation
						observation_time_window = random.randint(self.observation_time_window_range[0], self.observation_time_window_range[1])
						# pick a random start time for the observation within the request availability window
						observation_start = random.randint(request.time_start, request.time_end - observation_time_window)
						observation_end = observation_start + observation_time_window

						# CAN NON-EXCLUSIVE OBSERVATIONS BE ANYWHERE ? (paper interpretation is unclear)
						# -> basé sur la discussion avec le prof ainsi que le mail, je comprends que les observations non-exclusives ne peuvent pas chevaucher les créneaux exclusifs et non-exclusifs à la fois
						if not self.non_exclusive_observations_anywhere:
							# check if the observation time window is valid (either fully within an exclusive time or fully within a non-exclusive time)
							if not check_overlap_between_exclusive_and_normal_times(satellite, observation_start, observation_end):
								continue	# invalid observation time window, try again

						# create the observation
						observation = Observation(
							name = f"o_0_{request.get_id()}_{obs_index}",
							time_start = observation_start,
							time_end = observation_end,
							duration = request.duration,
							request = request,
							reward = request.reward,
							satellite = satellite
						)
						observations[observation.name] = observation
						request.observations.append(observation)
						success = True
					else:
						# Exclusive user request
						# pick one of the user's exclusive times
						exclusive_windows_for_user = request.user.exclusive_windows
						if not exclusive_windows_for_user:
							break	# no exclusive windows for this user, cannot create observation
						exclusive_window = random.choice(exclusive_windows_for_user)
						satellite = exclusive_window.satellite
						# pick a random time window for the observation within the exclusive time
						observation_time_window = random.randint(self.observation_time_window_range[0], self.observation_time_window_range[1])
						# pick a random start time for the observation within the exclusive time
						if exclusive_window.time_end - exclusive_window.time_start < observation_time_window:
							continue	# exclusive time too short for the observation, try again
						observation_start = random.randint(exclusive_window.time_start, exclusive_window.time_end - observation_time_window)
						observation_end = observation_start + observation_time_window
						# create the observation
						observation = Observation(
							name = f"o_{request.user.get_id()}_{request.get_id()}_{obs_index}",
							time_start = observation_start,
							time_end = observation_end,
							duration = request.duration,
							request = request,
							reward = request.reward,
							satellite = satellite
						)
						observations[observation.name] = observation
						request.observations.append(observation)
						success = True
				if not success:
					print(f"Warning: Could not create observation for request {request.name} after {attempts_allowed} random assignement attempts.")

		return observations

	def generate_instance(self) -> Problem:
		# generates a random instance of the problem based on the parameters
		
		# 1. generating the satellites
		satellites = self.generate_satellites()

		# 2. generating the users
		users = self.generate_users()

		# 3. generating the user exclusive times
		if self.exclusive_windows_generation_method == "paper":
			self.generate_exclusive_windows_paper_version(satellites, users)
		else:
			self.generate_exclusive_windows_subject_version(satellites, users)

		# 4. generating the requests for each user
		requests = self.generate_requests(users)

		# 5. generating the observations for each request
		observations = self.generate_observations(satellites, requests)

		print(f"Generated {len(satellites)} satellites, {len(users)} users, {len(requests)} requests, and {len(observations)} observations.")
		if self.verbose_results:
			print("Satellites:")
			for satellite in satellites.values():
				print(f"  {satellite.name}: time [{satellite.time_start}, {satellite.time_end}], capacity {satellite.capacity}, transition time {satellite.transition_time}, exclusive times {len(satellite.exclusive_windows)}")
			print("Users:")
			for user in users.values():
				print(f"  {user.name}: priority {user.priority}, exclusive times {len(user.exclusive_windows)}")
			print("Requests:")
			for request in requests.values():
				print(f"  {request.name}: user {request.user.name}, time [{request.time_start}, {request.time_end}], duration {request.duration}, reward {request.reward}, observations {len(request.observations)}")
			print("Observations:")
			for observation in observations.values():
				print(f"  {observation.name}: request {observation.request.name}, satellite {observation.satellite.name}, time [{observation.time_start}, {observation.time_end}], duration {observation.duration}, reward {observation.reward}")

		problem = Problem(
			satellites = satellites,
			users = users,
			requests = requests,
			observations = observations
		)
		return problem

## Visualisation

In [ ]:
# ------------------------------ VISUALIZATION ------------------------------
def plot_instance(problem: Problem, solution: Solution = None, t_min=None, t_max=None, display_in_browser: bool = True, title: str = ""):
	satellites = problem.satellites
	observations = problem.observations
	users = problem.users

	print(f"\nPlotting instance with {len(satellites)} satellites and {len(observations)} observations.")
	print("  this will open a browser window with an interactive plotly plot.")
	if len(observations) > 100:
		print("  Large instance, this may take some time...")
	
	if t_min is None:
		t_min = satellites["s_0"].time_start
	if t_max is None:
		t_max = satellites["s_0"].time_end
	
	# Couleurs par utilisateur
	user_colors = {
		user.name: f"hsl({(i * 60) % 360},70%,50%)"
		for i, user in enumerate(users.values())
	}
	selected_color = "black"

	fig = go.Figure()

	y_ticks = []
	y_labels = []

	current_y = 0
	satellite_y_bounds = {}

	if solution :
		solution_schedules = solution.get_solution_schedules()

	# ---------
	# 1) Construire l'axe Y par satellite
	# ---------
	for sat in satellites.values():
		sat_obs = [o for o in observations.values() if o.satellite.name == sat.name]

		if not sat_obs:
			continue

		y_start = current_y

		for obs in sat_obs:
			y_ticks.append(current_y)
			y_labels.append(f"{sat.name} | {obs.name}")

			fig.add_trace(go.Bar(
				x=[obs.time_end - obs.time_start],
				y=[current_y],
				base=[obs.time_start],
				orientation="h",
				marker=dict(color=user_colors[obs.user.name]),
				text=[obs.name],
				textposition="inside",
				insidetextanchor="middle",
				hovertemplate=(
					f"<b>{obs.name}</b><br>"
					f"Satellite: {sat.name}<br>"
					f"User: {obs.user.name}<br>"
					f"Time: [{obs.time_start}, {obs.time_end}]<extra></extra>"
				),
				showlegend=False
			))

			if solution and obs.name in solution_schedules:
				sol_start, sol_end = solution_schedules[obs.name]

				fig.add_shape(
					type="rect",
					x0=sol_start,
					x1=sol_end,
					y0=current_y - 0.4,
					y1=current_y + 0.4,
					# fillcolor=user_colors[obs.user.name],
					fillcolor=selected_color,
					opacity=0.5,
					# darker color border
					line=dict(color=selected_color, width=1),
					# line=dict(color=user_colors[obs.user.name], width=3),
					layer="above"
				)



			current_y += 1

		y_end = current_y - 1
		satellite_y_bounds[sat.name] = (y_start, y_end)

		# espace visuel entre satellites
		current_y += 1

	# ---------
	# 2) Créneaux exclusifs (fond de section)
	# ---------
	for sat in satellites.values():
		if sat.name not in satellite_y_bounds:
			continue

		y0, y1 = satellite_y_bounds[sat.name]

		for window in sat.exclusive_windows:
			fig.add_shape(
				type="rect",
				x0=window.time_start,
				x1=window.time_end,
				y0=y0 - 0.5,
				y1=y1 + 0.5,
				fillcolor=user_colors[window.user.name],
				opacity=0.3,
				line_width=0,
				layer="below"
			)

	# ---------
	# 3) Mise en forme générale
	# ---------
	fig.update_layout(
		title="Satellite Observation Scheduling Instance" + (f" - {title}" if title else ""),
		xaxis=dict(
			title="Time",
			range=[t_min, t_max],
			fixedrange=False
		),
		yaxis=dict(
			tickmode="array",
			tickvals=y_ticks,
			ticktext=y_labels,
			autorange="reversed"
		),
		height=max(800, current_y * 20),
		bargap=0.1,
		margin=dict(l=300, r=50, t=50, b=50)
	)

	if display_in_browser:
		fig.show(renderer="browser")
	else:
		# display in notebook
		fig.show()

# Algorithmes

## Greedy

In [ ]:
# ------------------------------ GREEDY SOLVER IMPLEMENTATION ------------------------------
# Algorithm 1: Greedy EOSCSP solver
def first_slot(observation: Observation, satellites_schedule: dict):
	# find the first available time slot on the satellite for the observation
	# considering the satellite's existing schedule, transition times, // and capacity
	# returns the start time of the first available slot, or None if no slot is available
	# not exactly the same implementation as in the paper, but similar idea I think

	# # check if the satellite is already at max capacity
	# satellite_schedule = satellites_schedule[observation.satellite.name]
	# if satellite_capacity is None:
	# 	satellite_capacity = observation.satellite.capacity
	# if len(satellite_schedule) >= satellite_capacity:
	# 	return None

	# get the variables
	satellite = observation.satellite
	schedule = satellites_schedule[satellite.name]
	observation_duration = observation.request.duration
	observation_start_window = observation.time_start
	observation_end_window = observation.time_end

	# add the transition times to the start and end of each scheduled observation
	extended_schedule = []
	for start, end, obs_name in schedule:
		extended_schedule.append((start - satellite.transition_time, end + satellite.transition_time, obs_name))

	# sort the extended schedule by start time
	extended_schedule.sort(key=lambda x: x[0])

	# scan for gaps
	current_time = observation_start_window
	for start, end, obs_name in extended_schedule:
		# Ignore tasks completely outside the window
		if end <= observation_start_window or start >= observation_end_window:
			continue
		
		# Clamp to window
		start = max(start, observation_start_window)
		end = min(end, observation_end_window)

		# Check gap
		if current_time + observation_duration <= start:
			return current_time

		# Move current time forward
		current_time = max(current_time, end)

	# Check after last scheduled observation
	if current_time + observation_duration <= observation_end_window:
		return current_time
	return None

def greedy_solver(problem: Problem, satellites_remaining_capacity: dict = None) -> Solution:
	# Greedy solver implementation from the paper
	# satellites_remaining_capacity : optional dict of remaining capacities per satellite (if None, use full capacities)

	scheduled_observations = []	# solution's scheduled observations
	# sort observations in increasing order on priority and start time criteria 
	observations : list[Observation] = list(problem.observations.values())
	observations_sorted : list[Observation] = sorted(observations, key=lambda obs: (obs.priority, -obs.reward))
	# observations_sorted = sorted(problem.observations.items(), key=lambda item: (item[1].priority, -item[1].reward))
	satellites_schedules = {sat_name: [] for sat_name in problem.satellites.keys()}
	if satellites_remaining_capacity is None:
		satellites_remaining_capacity = {sat_name: problem.satellites[sat_name].capacity for sat_name in problem.satellites.keys()}

	# for each observation 
	# can't use a for loop since we need to remove observations at the same time
	while observations_sorted:
		# pop the first observation
		observation = observations_sorted.pop(0)

		# check if the satellite has remaining capacity
		if satellites_remaining_capacity[observation.satellite.name] <= 0:
			continue	# skip this observation
		
		# find the first available time slot on the satellite for the observation
		start_time = first_slot(observation, satellites_schedules)
		if start_time is not None:
			# schedule the observation
			scheduled_observations.append((observation, start_time))
			# update the satellite's schedule
			end_time = start_time + observation.request.duration
			satellites_schedules[observation.satellite.name].append((start_time, end_time, observation.name))
			satellites_remaining_capacity[observation.satellite.name] -= 1
			# remove all other observations for the same request (since each request can be fulfilled at most once)
			for other_observation in list(observations_sorted):
				if other_observation.request.name == observation.request.name:
					observations_sorted.remove(other_observation)

	return Solution(
		problem = problem,
		scheduled_observations = scheduled_observations
	)

# Data

In [ ]:
# ------------------------------ INPUT DATA (PAPER) ------------------------------
# values from the paper's instance (cf 6.EXPERIMENTAL EVALUATION section in paper)
# By default they are overwritten by the next code section

# General :
computing_time: float = None	# Total time available for computing the solution

# Satellites :
number_of_satellites: int = 3	# Total number of satellites
capacity_per_satellite: int = 20	# Maximum number of observations doable per satellite over the period
satellites_availability_window : tuple = (0, 300)	# Time window for satellite availability (not randomized here)
transition_time_per_satellite: int = 1	# Transition time between 2 observations for each satellite (not randomized here)

# Users :
number_of_exclusive_users: int = 4	# Total number of users with exclusive access to certain satellites
exclusive_portions_per_user: int = 8	# Number of exclusive time portions per user (any satellite or per satellite ?)
exclusive_portion_duration_range: tuple = (15, 20)	# Duration range for each exclusive time portion

# Requests :
number_of_non_exclusive_requests_range: tuple = (2, 20)	# Range for the number of non-exclusive requests (all non-exclusive requests)
number_of_exclusive_requests_per_exclusive_user_range: tuple = (1, 5)	# Range for the number of exclusive requests per exclusive user
request_duration: int = 5	# Duration required to fulfill each request (not randomized here)
request_availability_window: tuple = (0, 300)	# Time window for each request availability (not randomized here (TODO: correct ?))
reward_non_exclusive_request_range: tuple = (1, 5)	# Reward range for non-exclusive requests
reward_exclusive_request_range: list = [10, 20, 30, 40, 50]	# Reward range for exclusive requests

# Observations :
number_of_observations_per_request = 3	# Number of possible observations per request (not randomized here)
# duration = request_duration	# Duration of each observation (same as request duration)
# reward = reward of the request	# Reward of each observation (same as request reward)
observation_time_window_range: tuple = (10, 20)	# Time window range for each observation (not randomized here)

random_instance_generator_paper = RandomInstanceGenerator(
	computing_time = computing_time,
	number_of_satellites = number_of_satellites,
	capacity_per_satellite = capacity_per_satellite,
	satellites_availability_window = satellites_availability_window,
	transition_time_per_satellite = transition_time_per_satellite,
	number_of_exclusive_users = number_of_exclusive_users,
	exclusive_portions_per_user = exclusive_portions_per_user,
	exclusive_portion_duration_range = exclusive_portion_duration_range,
	number_of_non_exclusive_requests_range = number_of_non_exclusive_requests_range,
	number_of_exclusive_requests_per_exclusive_user_range = number_of_exclusive_requests_per_exclusive_user_range,
	request_duration = request_duration,
	request_availability_window = request_availability_window,
	reward_non_exclusive_request_range = reward_non_exclusive_request_range,
	reward_exclusive_request_range = reward_exclusive_request_range,
	number_of_observations_per_request = number_of_observations_per_request,
	observation_time_window_range = observation_time_window_range
)

random_instance_generator_paper.exclusive_windows_generation_method = "paper"
random_instance_generator_paper.non_exclusive_observations_anywhere = True	# False according to paper interpretation, mais True est intréssant aussi

In [ ]:
# ------------------------------ INPUT DATA (adjusted from paper) ------------------------------
# new values choosen arbitrarily for testing
# -> intuition : based on the paper's values, but with less exclusive users to fit the subject's version
# -> to compensate, the number of non-exclusive requests is increased to keep the same order of magnitude of total requests

# General :
computing_time: float = None	# Total time available for computing the solution

# Satellites :
number_of_satellites: int = 3	# Total number of satellites
capacity_per_satellite: int = 20	# Maximum number of observations doable per satellite over the period
satellites_availability_window : tuple = (0, 300)	# Time window for satellite availability (not randomized here)
transition_time_per_satellite: int = 1	# Transition time between 2 observations for each satellite (not randomized here)

# Users :
number_of_exclusive_users: int = int(4/2)	# Total number of users with exclusive access to certain satellites
exclusive_portions_per_user: int = 8	# Number of exclusive time portions per user (any satellite)
exclusive_portion_duration_range: tuple = (15, 20)	# Duration range for each exclusive time portion

# Requests :
number_of_non_exclusive_requests_range: tuple = (8+(2*2), 80+(2*20))	# Range for the number of non-exclusive requests (all non-exclusive requests)
number_of_exclusive_requests_per_exclusive_user_range: tuple = (2, 20)	# Range for the number of exclusive requests per exclusive user
request_duration: int = 5	# Duration required to fulfill each request (not randomized here)
request_availability_window: tuple = (0, 300)	# Time window for each request availability (not randomized here (TODO: correct ?))
reward_non_exclusive_request_range: tuple = (1, 5)	# Reward range for non-exclusive requests
reward_exclusive_request_range: list = [10, 20, 30, 40, 50]	# Reward range for exclusive requests

# Observations :
number_of_observations_per_request = 10	# Number of possible observations per request (not randomized here)
# duration = request_duration	# Duration of each observation (same as request duration)
# reward = reward of the request	# Reward of each observation (same as request reward)
observation_time_window_range: tuple = (10, 20)	# Time window range for each observation (not randomized here)

random_instance_generator = RandomInstanceGenerator(
	computing_time = computing_time,
	number_of_satellites = number_of_satellites,
	capacity_per_satellite = capacity_per_satellite,
	satellites_availability_window = satellites_availability_window,
	transition_time_per_satellite = transition_time_per_satellite,
	number_of_exclusive_users = number_of_exclusive_users,
	exclusive_portions_per_user = exclusive_portions_per_user,
	exclusive_portion_duration_range = exclusive_portion_duration_range,
	number_of_non_exclusive_requests_range = number_of_non_exclusive_requests_range,
	number_of_exclusive_requests_per_exclusive_user_range = number_of_exclusive_requests_per_exclusive_user_range,
	request_duration = request_duration,
	request_availability_window = request_availability_window,
	reward_non_exclusive_request_range = reward_non_exclusive_request_range,
	reward_exclusive_request_range = reward_exclusive_request_range,
	number_of_observations_per_request = number_of_observations_per_request,
	observation_time_window_range = observation_time_window_range
)

In [ ]:
# ------------------------------ INPUT DATA (paper example adjusted) ------------------------------
# values choosen to create a smaller instance similar to the paper's example

# General :
computing_time: float = None	# Total time available for computing the solution

# Satellites :
number_of_satellites: int = 3	# Total number of satellites
capacity_per_satellite: int = 4	# Maximum number of observations doable per satellite over the period
satellites_availability_window : tuple = (20, 100)	# Time window for satellite availability (not randomized here)
transition_time_per_satellite: int = 1	# Transition time between 2 observations for each satellite (not randomized here)

# Requests :
request_duration: int = 8	# Duration required to fulfill each request (not randomized here)
request_availability_window: tuple = (0, 300)	# Time window for each request availability (not randomized here (TODO: correct ?))
reward_non_exclusive_request_range: tuple = (1, 5)	# Reward range for non-exclusive requests
reward_exclusive_request_range: list = [10, 20, 30, 40, 50]	# Reward range for exclusive requests

# 1. generating the satellites
satellites = {}
for i in range(number_of_satellites):
	satellite = Satellite(
		name = f"s_{i}",
		time_start = satellites_availability_window[0],
		time_end = satellites_availability_window[1],
		capacity = capacity_per_satellite,
		transition_time = transition_time_per_satellite
	)
	satellites[satellite.name] = satellite

# 2. generating the users
# 2.1 generating the exclusive users
users = {}
user_ex1 = User(
	name = "u_1",
	exclusive_windows = [],	# to be filled in step 3.
	priority = 1
)
users[user_ex1.name] = user_ex1
user_ex2 = User(
	name = "u_2",
	exclusive_windows = [],	# to be filled in step 3.
	priority = 2
)
users[user_ex2.name] = user_ex2
# 2.2 generating non-exclusive users (all aggregated into a single user)
non_exclusive_user = User(
	name = "u_0",
	exclusive_windows = [],	# no exclusive times for non-exclusive user
	priority = 3	# priority after all exclusive users
)
users[non_exclusive_user.name] = non_exclusive_user

# 3. generating the user exclusive times
# user_ex1 exclusive times : 20-35 and 40-60 on satellite s_0, 35-45 on satellite s_1
# user_ex2 exclusive times : 45-55 and 70-90 on satellite s_2
satellites["s_0"].add_exclusive_time(user_ex1, 20, 35)
user_ex1.exclusive_windows.append((satellites["s_0"], (20, 35)))
satellites["s_0"].add_exclusive_time(user_ex1, 40, 60)
user_ex1.exclusive_windows.append((satellites["s_0"], (40, 60)))
satellites["s_1"].add_exclusive_time(user_ex1, 35, 45)
user_ex1.exclusive_windows.append((satellites["s_1"], (35, 45)))
satellites["s_2"].add_exclusive_time(user_ex2, 45, 55)
user_ex2.exclusive_windows.append((satellites["s_2"], (45, 55)))
satellites["s_2"].add_exclusive_time(user_ex2, 70, 90)
user_ex2.exclusive_windows.append((satellites["s_2"], (70, 90)))

# 4. generating the requests for each user
# 4.1 generating non-exclusive requests
requests = {}
for i in range(11):	# 11 non-exclusive requests
	request_start = request_availability_window[0]
	request_end = request_availability_window[1]
	request = Request(
		name = f"r_0_{i}",
		time_start = request_start,
		time_end = request_end,
		duration = request_duration,
		reward = random.randint(reward_non_exclusive_request_range[0], reward_non_exclusive_request_range[1]),
		position = None,	# position generation not implemented yet
		user = users["u_0"],
		observations = []	# to be filled in step 5.
	)
	requests[request.name] = request
# 4.2 generating exclusive requests for each exclusive user
for u in range(2):	# 2 exclusive users
	for i in range(3):	# 3 exclusive requests
		if u == 1 and i == 2:
			break	# user 2 has only 2 exclusive requests
		request_start = request_availability_window[0]
		request_end = request_availability_window[1]
		request = Request(
			name = f"r_{u+1}_{i}",
			time_start = request_start,
			time_end = request_end,
			duration = request_duration,
			reward = random.choice(reward_exclusive_request_range),
			position = None,	# position generation not implemented yet
			user = users[f"u_{u+1}"],
			observations = []	# to be filled in step 5.
		)
		requests[request.name] = request

# 5. generating the observations for each request
observations = {}
def add_observation(user_id: int, request_id: int, obs_id: int, sat_id: int, time_start: int, time_end: int):
	obs = Observation(
		name = f"o_{user_id}_{request_id}_{obs_id}",
		time_start = time_start,
		time_end = time_end,
		duration = request_duration,
		request = requests[f"r_{user_id}_{request_id}"],
		reward = requests[f"r_{user_id}_{request_id}"].reward,
		satellite = satellites[f"s_{sat_id}"]
	)
	observations[obs.name] = obs
	requests[f"r_{user_id}_{request_id}"].observations.append(obs)
# satellites observations for each request
add_observation(user_id=0, request_id=0, obs_id=0, sat_id=0, time_start=20, time_end=30)
add_observation(user_id=0, request_id=1, obs_id=0, sat_id=0, time_start=25, time_end=35)
add_observation(user_id=0, request_id=2, obs_id=0, sat_id=0, time_start=25, time_end=35)
add_observation(user_id=0, request_id=3, obs_id=0, sat_id=0, time_start=45, time_end=55)
add_observation(user_id=0, request_id=4, obs_id=1, sat_id=0, time_start=25, time_end=35)	# edited
add_observation(user_id=0, request_id=7, obs_id=1, sat_id=0, time_start=25, time_end=35)	# edited
add_observation(user_id=1, request_id=0, obs_id=1, sat_id=0, time_start=25, time_end=35)
add_observation(user_id=1, request_id=1, obs_id=0, sat_id=0, time_start=40, time_end=60)
add_observation(user_id=1, request_id=1, obs_id=0, sat_id=0, time_start=40, time_end=60)	# new
add_observation(user_id=1, request_id=2, obs_id=0, sat_id=0, time_start=40, time_end=55)	# new

add_observation(user_id=0, request_id=0, obs_id=1, sat_id=1, time_start=35, time_end=45)	# edited
add_observation(user_id=0, request_id=2, obs_id=1, sat_id=1, time_start=25, time_end=35)
add_observation(user_id=0, request_id=3, obs_id=1, sat_id=1, time_start=45, time_end=55)
add_observation(user_id=0, request_id=5, obs_id=1, sat_id=1, time_start=60, time_end=80)
add_observation(user_id=0, request_id=6, obs_id=1, sat_id=1, time_start=70, time_end=80)
add_observation(user_id=1, request_id=0, obs_id=0, sat_id=1, time_start=35, time_end=45)

add_observation(user_id=0, request_id=1, obs_id=1, sat_id=2, time_start=45, time_end=55)
add_observation(user_id=0, request_id=4, obs_id=0, sat_id=2, time_start=30, time_end=40)
add_observation(user_id=0, request_id=5, obs_id=0, sat_id=2, time_start=70, time_end=90)	# edited
add_observation(user_id=0, request_id=6, obs_id=0, sat_id=2, time_start=80, time_end=90)
add_observation(user_id=0, request_id=7, obs_id=0, sat_id=2, time_start=45, time_end=55)	# edited
add_observation(user_id=0, request_id=8, obs_id=0, sat_id=2, time_start=80, time_end=90)	# new
add_observation(user_id=0, request_id=9, obs_id=0, sat_id=2, time_start=80, time_end=90)	# new
add_observation(user_id=0, request_id=10, obs_id=0, sat_id=2, time_start=80, time_end=90)	# new
add_observation(user_id=2, request_id=0, obs_id=1, sat_id=2, time_start=45, time_end=55)
add_observation(user_id=2, request_id=1, obs_id=1, sat_id=2, time_start=45, time_end=55)
add_observation(user_id=2, request_id=0, obs_id=0, sat_id=2, time_start=70, time_end=90)	# new
add_observation(user_id=2, request_id=1, obs_id=0, sat_id=2, time_start=70, time_end=90)	# new

problem_example = Problem(
	satellites = satellites,
	users = users,
	requests = requests,
	observations = observations
)

In [ ]:
# ------------------------------ INPUT DATA (new example) ------------------------------
# values choosen to create a smaller instance similar to the paper's example

# General :
computing_time: float = None	# Total time available for computing the solution

# Satellites :
number_of_satellites: int = 3	# Total number of satellites
capacity_per_satellite: int = 3	# Maximum number of observations doable per satellite over the period
satellites_availability_window : tuple = (0, 60)	# Time window for satellite availability (not randomized here)
transition_time_per_satellite: int = 1	# Transition time between 2 observations for each satellite (not randomized here)

# Requests :
request_duration: int = 8	# Duration required to fulfill each request (not randomized here)
request_availability_window: tuple = (0, 100)	# Time window for each request availability (not randomized here (TODO: correct ?))
reward_non_exclusive_request_order: tuple = [5, 4, 3, 2, 1]	# Reward range for non-exclusive requests
reward_exclusive_request_order: list = [50, 40, 30, 20, 10]	# Reward range for exclusive requests

# 1. generating the satellites
satellites = {}
for i in range(number_of_satellites):
	satellite = Satellite(
		name = f"s_{i}",
		time_start = satellites_availability_window[0],
		time_end = satellites_availability_window[1],
		capacity = capacity_per_satellite,
		transition_time = transition_time_per_satellite
	)
	satellites[satellite.name] = satellite

# 2. generating the users
# 2.1 generating the exclusive users
users = {}
user_ex1 = User(
	name = "u_1",
	exclusive_windows = [],	# to be filled in step 3.
	priority = 1
)
users[user_ex1.name] = user_ex1
user_ex2 = User(
	name = "u_2",
	exclusive_windows = [],	# to be filled in step 3.
	priority = 2
)
users[user_ex2.name] = user_ex2
# 2.2 generating non-exclusive users (all aggregated into a single user)
non_exclusive_user = User(
	name = "u_0",
	exclusive_windows = [],	# no exclusive times for non-exclusive user
	priority = 3	# priority after all exclusive users
)
users[non_exclusive_user.name] = non_exclusive_user

# 3. generating the user exclusive times
# user_ex1 exclusive times : 20-35 and 40-60 on satellite s_0, 35-45 on satellite s_1
# user_ex2 exclusive times : 45-55 and 70-90 on satellite s_2
satellites["s_0"].add_exclusive_time(user_ex1, 0, 20)
user_ex1.exclusive_windows.append((satellites["s_0"], (0, 20)))
satellites["s_0"].add_exclusive_time(user_ex1, 30, 50)
user_ex1.exclusive_windows.append((satellites["s_0"], (30, 50)))
satellites["s_1"].add_exclusive_time(user_ex1, 0, 20)
user_ex1.exclusive_windows.append((satellites["s_1"], (0, 20)))
satellites["s_2"].add_exclusive_time(user_ex2, 0, 20)
user_ex2.exclusive_windows.append((satellites["s_2"], (0, 20)))
satellites["s_2"].add_exclusive_time(user_ex2, 30, 50)
user_ex2.exclusive_windows.append((satellites["s_2"], (30, 50)))

# 4. generating the requests for each user
# 4.1 generating non-exclusive requests
requests = {}
for i in range(5):	# 5 non-exclusive requests
	request_start = request_availability_window[0]
	request_end = request_availability_window[1]
	request = Request(
		name = f"r_0_{i}",
		time_start = request_start,
		time_end = request_end,
		duration = request_duration,
		reward = reward_non_exclusive_request_order[i],
		position = None,	# position generation not implemented yet
		user = users["u_0"],
		observations = []	# to be filled in step 5.
	)
	requests[request.name] = request
# 4.2 generating exclusive requests for each exclusive user
for u in range(2):	# 2 exclusive users
	for i in range(3):	# 3 exclusive requests
		if u == 1 and i == 2:
			break	# user 2 has only 2 exclusive requests
		request_start = request_availability_window[0]
		request_end = request_availability_window[1]
		request = Request(
			name = f"r_{u+1}_{i}",
			time_start = request_start,
			time_end = request_end,
			duration = request_duration,
			reward = reward_exclusive_request_order[u*3+i],
			position = None,	# position generation not implemented yet
			user = users[f"u_{u+1}"],
			observations = []	# to be filled in step 5.
		)
		requests[request.name] = request

# 5. generating the observations for each request
observations = {}
def add_observation(user_id: int, request_id: int, obs_id: int, sat_id: int, time_start: int, time_end: int):
	obs = Observation(
		name = f"o_{user_id}_{request_id}_{obs_id}",
		time_start = time_start,
		time_end = time_end,
		duration = request_duration,
		request = requests[f"r_{user_id}_{request_id}"],
		reward = requests[f"r_{user_id}_{request_id}"].reward,
		satellite = satellites[f"s_{sat_id}"]
	)
	observations[obs.name] = obs
	requests[f"r_{user_id}_{request_id}"].observations.append(obs)
# satellites observations for each request
add_observation(user_id=1, request_id=0, obs_id=0, sat_id=0, time_start=0, time_end=20)
add_observation(user_id=1, request_id=1, obs_id=0, sat_id=0, time_start=0, time_end=10)
add_observation(user_id=1, request_id=2, obs_id=0, sat_id=0, time_start=30, time_end=50)
add_observation(user_id=0, request_id=0, obs_id=0, sat_id=0, time_start=10, time_end=20)

add_observation(user_id=1, request_id=2, obs_id=1, sat_id=1, time_start=0, time_end=20)
add_observation(user_id=0, request_id=0, obs_id=1, sat_id=1, time_start=0, time_end=20)
add_observation(user_id=0, request_id=1, obs_id=0, sat_id=1, time_start=0, time_end=20)
add_observation(user_id=0, request_id=1, obs_id=1, sat_id=1, time_start=30, time_end=40)
add_observation(user_id=0, request_id=2, obs_id=0, sat_id=1, time_start=30, time_end=50)
add_observation(user_id=0, request_id=3, obs_id=0, sat_id=1, time_start=40, time_end=50)
add_observation(user_id=0, request_id=4, obs_id=0, sat_id=1, time_start=40, time_end=50)

add_observation(user_id=2, request_id=0, obs_id=0, sat_id=2, time_start=0, time_end=10)
add_observation(user_id=2, request_id=0, obs_id=1, sat_id=2, time_start=30, time_end=40)
add_observation(user_id=2, request_id=1, obs_id=0, sat_id=2, time_start=0, time_end=10)
add_observation(user_id=2, request_id=1, obs_id=1, sat_id=2, time_start=30, time_end=40)
add_observation(user_id=0, request_id=4, obs_id=1, sat_id=2, time_start=40, time_end=50)



new_example_problem_example = Problem(
	satellites = satellites,
	users = users,
	requests = requests,
	observations = observations
)

# Execution - Greedy Solver

In [ ]:
# ------------------------------ MAIN EXECUTION FOR GREEDY ------------------------------
# problem = random_instance_generator_paper.generate_instance()
problem = random_instance_generator.generate_instance()
# problem = problem_example
# problem = new_example_problem_example

solution = greedy_solver(problem)

is_valid = solution.verify()
total_reward = solution.total_reward()
proportion_of_scheduled_requests, n_completed_requests, n_all_requests = solution.get_proportion_of_completed_requests()

print(f"Solution valid: {is_valid}")
print(f"Total reward of the solution: {total_reward}")
print(f"Scheduled requests: {proportion_of_scheduled_requests*100:.2f}% ({n_completed_requests} out of {n_all_requests})")

plot_instance(problem, solution, display_in_browser=True, title="Greedy Solution")	# time window of the example

# Partie 1 : Optimisation de contraintes distribuées

In [ ]:
# ------------------------------- PYDCOP FUNCTIONS -------------------------------
def convert_DCOP_to_yaml(dcop : dict, variables_mapping : dict) -> str:
	# function to convert the DCOP instance to a yaml string compatible with PyDcop
	yaml_content = ""

	yaml_content += f"name: '{dcop['name']}'\n"
	yaml_content += f"objective: {dcop['objective']}\n\n"

	# Domains:
	yaml_content += "domains: \n"
	for domain in dcop['domains']:
		yaml_content += f"  {domain['name']}:\n"
		# values
		values_list = ", ".join([str(v) for v in domain['values']])
		yaml_content += f"    values: [{values_list}]\n"
		# type
		if domain['domain_type'] is not None:
			yaml_content += f"    type: {domain['domain_type']}\n"
	yaml_content += "\n"

	yaml_content += "variables:\n"
	for variable in dcop['variables']:
		yaml_content += f"  {variable['name']}:\n"
		yaml_content += f"	domain: {variable['domain']['name']}\n"
	yaml_content += "\n"

	yaml_content += "constraints:\n"
	for constraint_str in dcop['constraints']:
		yaml_content += constraint_str
	yaml_content += "\n\n"

	yaml_content += "agents:\n"
	for agent_name in dcop['agents']:
		variables_mapping_for_agent = [var for var in variables_mapping if variables_mapping[var]['agent_name'] == agent_name]
		yaml_content += f"  {agent_name}:\n"
		yaml_content += f"	variables: [{', '.join([var for var in variables_mapping_for_agent])}]\n"
	# add auxiliary agents because PyDcop seemingly requires at least 1 agent per variable
	number_of_variables = len(dcop['variables'])
	number_of_agents = len(dcop['agents'])
	if number_of_variables > number_of_agents:
		for i in range(number_of_agents, number_of_variables):
			aux_agent_name = f"aux_agent_{i}"
			variable_name = dcop['variables'][i]['name']
			yaml_content += f"  {aux_agent_name}:\n"
			# yaml_content += f"	variables: [{variable_name}]\n"

	return yaml_content

def write_yaml_file(yaml_content: str, keep_file: bool = False) -> str:
	# function to write the yaml content to a file
	# for debugging purposes keep the file after execution
	if keep_file:
		file_path = "dcop_instance.yaml"
		with open(file_path, "w") as yaml_file:
			yaml_file.write(yaml_content)
		return file_path
	with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as yaml_file:
		yaml_file.write(yaml_content)
		return yaml_file.name	# return the path to the temporary file

def pydcop_solve(yaml_file_path: str, pydcop_algorithm: str, timeout: int = None, verbose: bool = False) -> dict:
	# function to solve the problem using PyDcop, makes a call in the terminal to run the pydcop command, then returns the solution printed in the terminal
	try:
		cmd = "pydcop"
		if timeout is not None:
			cmd += f" --timeout {timeout}"	# TODO: test if timeout actually works
		cmd += " solve"
		cmd += f" --algo {pydcop_algorithm}"
		cmd += f" {yaml_file_path}"
		result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
		
		if result.returncode != 0:
			# test if it failed because of timeout
			if "timeout" in result.stderr:	# TODO: check exact error message when timeout occurs
				if verbose:
					print("PyDcop command timed out.")
			else:
				print(f"PyDcop command failed with return code {result.returncode}")
				print("Error message:", result.stderr)
			return None

		if verbose:
			print("PyDcop output:", result.stdout)
		if result.stdout:
			pydcop_output = json.loads(result.stdout)

			# TODO: test if these fields works with the implemented algorithms
			solution_data = None
			if "assignment" in pydcop_output:
				solution_data = pydcop_output["assignment"]
			elif "solution" in pydcop_output:
				solution_data = pydcop_output["solution"]
			else:
				print("No solution found in PyDcop output.")
			if solution_data is not None:
				if verbose:
					print("Solution data from PyDcop:", solution_data)
				return solution_data
		else:
			if verbose:
				print("No output from PyDcop.")
		return None
	except Exception as e:
		print(f"An error occurred while running PyDcop: {e}")

# ------------------------------ HELPER FUNCTIONS ------------------------------
def combine_solutions(solutions : list[Solution], problem : Problem) -> Solution:
	# combine multiple solutions into a single solution
	combined_scheduled_observations = []
	for sol in solutions:
		combined_scheduled_observations.extend(sol.scheduled_observations)
	combined_scheduled_observations = list(set(combined_scheduled_observations))	# remove duplicates
	return Solution(
		problem = problem,
		scheduled_observations = combined_scheduled_observations
	)

In [ ]:
# CODE TO TEST IF PYDCOP WORKS
yaml_content = """
name: graph coloring
objective: min

domains:
  colors:
	values: [R, G]

variables:
  v1:
	domain: colors
  v2:
	domain: colors
  v3:
	domain: colors

constraints:
    pref_1:
      type: extensional
      variables: v1
      values:
        -0.1: R
        0.1: G

    pref_2:
      type: extensional
      variables: v2
      values:
        -0.1: G
        0.1: R

    pref_3:
      type: extensional
      variables: v3
      values:
        -0.1: G
        0.1: R

    diff_1_2:
      type: intention
      function: 10 if v1 == v2 else 0

    diff_2_3:
      type: intention
      function: 10 if v3 == v2 else 0

agents: [a1, a2, a3, a4, a5]

"""
yaml_file_path = write_yaml_file(yaml_content)
start_time = time.time()
solution = pydcop_solve(yaml_file_path, "dpop", timeout=1, verbose=True)  # not sure if the timeout is working, it only takes seconds, and the test problem is faster than 1 second
print(f"PyDcop solving time: {time.time() - start_time:.4f} seconds") # (it's not faster than 1s (maybe initialization overhead ?))
print("Combined PyDcop solution:", solution)

In [ ]:
# ------------------------------ DCOP BUILDING FUNCTION ------------------------------
def get_DCOP_parameters(request : Request, non_exclusive_solution : Solution, exclusive_users_assignements : dict, problem : Problem, satellites_remaining_capacity : dict) -> tuple[dict, dict, dict]:
	# function to build the parameters of a DCOP instance from a request and its assignements
	# returns the DCOP parameters, a dict of {variable_name -> {agent_name, user_id, satellite_name, observation_name}}, and a dict of {observation.name -> start_time} to avoid recomputing the best start time of the assigned observation
	dcop_name = f"DCOP_request_{request.get_id()}"
	dcop_objective = "max"	# maximize total reward added
	
	# 1) create agents : exclusive users who can potentially fulfill an observation of the request
	agents_potential_observations = {}	# dict of user_id -> {observation_name -> (Observation, ExclusiveWindow)}
	for observation in request.observations:
		for window in observation.satellite.exclusive_windows:
			if window.can_potentially_assign_observation(observation):
				# add the observation to the user's potential observations
				if window.user.get_id() not in agents_potential_observations:
					agents_potential_observations[window.user.get_id()] = {}
				agents_potential_observations[window.user.get_id()][observation.name] = (observation, window)
	agents = [agent.name for agent in problem.users.values() if agent.get_id() in agents_potential_observations]
	# create the AgentDef for each agent
	# agents = {}
	# for user_id, user_observations in agents_potential_observations.items():
	# 	agents[user_id] = AgentDef(name = f"agent_user_{user_id}")

	# 2) create domains : binary domain
	domains = []
	binary_domain = {"name" : "binary_domain", "domain_type": "bool", "values": [0, 1]}
	domains.append(binary_domain)
	# Domain(name="binary_domain", domain_type="bool", values=[0, 1])	# 0: not assigned, 1: assigned

	# 3) create decision variables : for each agent : create a variable for each potential observation
	variables_mapping = {}	# dict of {variable_name -> {agent_name -> user_id, satellite_name -> satellite.name, observation_name -> observation.name}}
	variables = []
	for user_id, potential_user_observations in agents_potential_observations.items():
		for observation_name, (observation, window) in potential_user_observations.items():
			var_name = f"decision_var_exclusive_win_{window.name}_observation_{observation_name}_user_{user_id}"
			# variable = Variable(
			# 	name = var_name,
			# 	domain = binary_domain
			# )
			variable = {"name": var_name, "domain": binary_domain}
			variables.append(variable)
			variables_mapping[var_name] = {"agent_name": problem.users[f"u_{user_id}"].name, "user_id": user_id, "satellite_name": observation.satellite.name, "observation_name": observation.name}

	# 4) create constraints : to be implemented
	infinity = -100000000
	constraints_str = []
	# 4.1) at most 1 observation assigned
	all_variable_names = [variable["name"] for variable in variables]
	variable_names_str = " + ".join(all_variable_names)
	contrainte1 = []
	contrainte1.append("  at_most_one_assigned_observation:")
	contrainte1.append("    type: intention")
	contrainte1.append("    function: 0 if " + variable_names_str + " <= 1 else " + str(infinity))
	constraints_str.append("\n".join(contrainte1))

	# 4.2) satellite capacity constraints
	# satellites_remaining_capacity = {}
	# for sat in problem.satellites.values():
	# 	satellites_remaining_capacity[sat.name] = sat.capacity
	# # reduce capacity based on current assignments in non_exclusive_solution and exclusive_users_assignements
	# # scheduled_observations = non_exclusive_solution.scheduled_observations
	# # scheduled_observations += [obs for user_solution in exclusive_users_assignements.values() for obs in user_solution.scheduled_observations]
	# for scheduled_observation, start_time in non_exclusive_solution.scheduled_observations:
	# 	sat_name = scheduled_observation.satellite.name
	# 	if sat_name in satellites_remaining_capacity:
	# 		satellites_remaining_capacity[sat_name] -= 1
	# exclusive_users_assignements = {user.name: user_solution for user, user_solution in exclusive_users_assignements.items()}
	# for scheduled_observation, start_time in combine_solutions(list(exclusive_users_assignements.values()), problem).scheduled_observations:
	# 	sat_name = scheduled_observation.satellite.name
	# 	if sat_name in satellites_remaining_capacity:
	# 		satellites_remaining_capacity[sat_name] -= 1

	for sat_name, remaining_capacity in satellites_remaining_capacity.items():
		# get all variables related to this satellite
		sat_variable_names = []
		for var_name, var_info in variables_mapping.items():
			if var_info["satellite_name"] == sat_name:
				sat_variable_names.append(var_name)
		if not sat_variable_names:
			continue
		variable_names_str = " + ".join(sat_variable_names)
		contrainte_sat = []
		contrainte_sat.append(f"\n  satellite_{sat_name}_capacity_constraint:")
		contrainte_sat.append("    type: intention")
		contrainte_sat.append("    function: 0 if " + variable_names_str + f" <= {remaining_capacity} else " + str(infinity))
		constraints_str.append("\n".join(contrainte_sat))

	# 4.3.0) save {observation.name -> start_time} to return it along with the DCOP
	assigned_observations_start_times = {}

	# 4.3) soft constraints
	# for each observation : look at the corresponding exclusive window's user's assignements
	for user_id, potential_observations in agents_potential_observations.items():
		user_name = f"u_{user_id}"
		for observation_name, (observation, window) in potential_observations.items():
			var_name = f"decision_var_exclusive_win_{window.name}_observation_{observation_name}_user_{user_id}"
			# get the exclusive user's assignement
			if user_name in exclusive_users_assignements:
				user_solution = exclusive_users_assignements[user_name]
				# check if the observation can be scheduled in the user's assignement
				satellite = observation.satellite
				satellite_schedule = user_solution.get_satellite_schedule(satellite)
				# start_time = first_slot(observation, {satellite.name: satellite_schedule})
				# if start_time is not None:
				# 	# create a soft constraint with cost = -reward if assigned
				# 	contrainte_soft = []
				# 	contrainte_soft.append(f"  soft_constraint_{var_name}:")
				# 	contrainte_soft.append("    type: intention")
				# 	contrainte_soft.append(f"    function: {observation.reward} if {var_name} == 1 else 0")
				# 	constraints.append("\n".join(contrainte_soft))
				# else :
				# test what the best score would be if the observation was forced to be scheduled
				extended_satellite_schedule = sort_and_extend_satellite_schedule(satellite, satellite_schedule)
				max_reward, best_start_time = get_best_observation_replacement(observation, extended_satellite_schedule, problem)
				# create a soft constraint with cost = -max_reward
				contrainte_soft = []
				contrainte_soft.append(f"\n  soft_constraint_{var_name}:")
				contrainte_soft.append("    type: intention")
				contrainte_soft.append(f"    function: {max_reward} if {var_name} == 1 else 0")
				constraints_str.append("\n".join(contrainte_soft))
				assigned_observations_start_times[observation.name] = best_start_time
	
	# 5) create the DCOP 
	dcop : dict = {}
	dcop["name"] = dcop_name
	dcop["objective"] = dcop_objective
	dcop["domains"] = domains
	dcop["variables"] = variables
	dcop["constraints"] = constraints_str
	dcop["agents"] = agents

	return dcop, variables_mapping, assigned_observations_start_times

def create_solution_from_dcop_return(dcop_solution : dict, variables_mapping : dict, assigned_observations_start_times : dict, problem : Problem) -> tuple[Observation, int]:
	# function to create a Solution object from the DCOP solution returned by PyDcop
	scheduled_observations = []
	for var_name, value in dcop_solution.items():
		if value == 1:
			var_info = variables_mapping[var_name]
			observation_name = var_info["observation_name"]
			observation = problem.observations[observation_name]
			# get the start time from assigned_observations_start_times
			start_time = assigned_observations_start_times[observation_name]
			scheduled_observations.append((observation, start_time))
	if len(scheduled_observations) > 1:
		print(f"Warning: more than one observation scheduled in DCOP solution: {scheduled_observations}")
	if len(scheduled_observations) == 0:
		return None
	return scheduled_observations[0]

# Execution - s_DCOP Solver

In [ ]:
# RESOLUTION s_dcop EOSCSP solver
# Steps :
# 0) input : instance du problème
# 1) pour chaque user exclusif :
# 	création de l'assignement des observations de l'utilisateur exclusif avec greedy()
# 3) création de l'assignement des observations hors des fenêtres exclusives avec greedy()	(changé l'ordre des étapes 2 et 3)
# 2) pour chaque requète (non exclusive) restante, dans l'ordre :
# 	création d'un DCOP avec tous les assignements créés jusqu'à présent ainsi que les observations de la requète actuelle
# 	la résolution du DCOP renvoie des nouvelles assignations pour les zones des utilisateurs exclusifs
# 4) Enfin on combine les assignements trouvés pour avoir la solution final

def update_satellite_capacity(scheduled_observations : list[tuple[Observation, int]], satellites_remaining_capacity : dict, remove: bool = False):
	# function to update the satellites_remaining_capacity based on the scheduled_observations
	for scheduled_observation, start_time in scheduled_observations:
		sat_name = scheduled_observation.satellite.name
		if sat_name in satellites_remaining_capacity:
			if remove:
				satellites_remaining_capacity[sat_name] += 1
			else:
				satellites_remaining_capacity[sat_name] -= 1

# problem = random_instance_generator_paper.generate_instance()
problem = random_instance_generator.generate_instance()
# problem = problem_example
# problem = new_example_problem_example

# 0) tracker les capacités des satellites restantes
satellites_remaining_capacity = {}
for sat in problem.satellites.values():
	satellites_remaining_capacity[sat.name] = sat.capacity

# 1) pour chaque user exclusif :
exclusive_users = problem.get_exclusive_users()
# sort exclusive users by priority
exclusive_users = dict(sorted(exclusive_users.items(), key=lambda item: item[1].priority))
exclusive_users_solutions = {}
for user in exclusive_users.values():
	# création de l'assignement des observations de l'utilisateur exclusif avec greedy()
	user_problem = Problem(
		satellites = problem.satellites,
		users = {user.name: user},
		requests = problem.get_user_requests(user),
		observations = problem.get_user_observations(user)
	)
	user_solution = greedy_solver(user_problem, satellites_remaining_capacity)
	# reduce capacity based on user_solution assignments
	# update_satellite_capacity(user_solution.scheduled_observations, satellites_remaining_capacity)
	exclusive_users_solutions[user.name] = user_solution
	print(f"User {user.name} exclusive observations scheduled: {len(user_solution.scheduled_observations)}")

# 2.0) combinaison des solutions partielles
initial_solution = combine_solutions(list(exclusive_users_solutions.values()), problem)
non_exclusive_requests_completed = []
# 2) pour chaque requète non assignée restante (non exclusive) :
non_exclusive_requests = problem.get_non_exclusive_requests()
uncompleted_requests = initial_solution.get_uncompleted_requests()
remaining_requests = {
	req_name: req
	for req_name, req in non_exclusive_requests.items()
	if req_name in uncompleted_requests
}
# sort remaining requests by reward descending
remaining_requests = dict(sorted(remaining_requests.items(), key=lambda item: item[1].reward, reverse=True))
for request in remaining_requests.values():
	# création d'un DCOP avec tous les assignements créés jusqu'à présent ainsi que les observations de la requète actuelle
	# la résolution du DCOP renvoie des nouvelles assignations pour les zones des utilisateurs exclusifs
	dcop, variables_mapping, assigned_observations_start_times = get_DCOP_parameters(request, None, exclusive_users_solutions, problem, satellites_remaining_capacity)
	yaml_content = convert_DCOP_to_yaml(dcop, variables_mapping)
	yaml_file_path = write_yaml_file(yaml_content, keep_file=True)
	dcop_return = pydcop_solve(yaml_file_path, "dpop", timeout=10, verbose=False)  # timeout of 10 seconds for each DCOP
	if dcop_return is None:
		print(f"No solution found for DCOP of request {request.name}, skipping to next request.")
		continue
	dcop_return = create_solution_from_dcop_return(dcop_return, variables_mapping, assigned_observations_start_times, problem)
	if dcop_return is None:
		print(f"No observation scheduled in DCOP solution for request {request.name}, skipping to next request.")
		continue
	dcop_observation, start_time = dcop_return
	# update the corresponding exclusive user's solution with the new assignments from the DCOP solution
	exclusive_user_doing_the_observation = get_exclusive_user_from_observation_assignment(dcop_observation)
	user_name = exclusive_user_doing_the_observation.name
	user_solution = exclusive_users_solutions[user_name]
	# remove any observation on the same satellite at overlapping times
	removed_observation = remove_overlapping_observations(dcop_observation, start_time, user_solution)
	update_satellite_capacity(removed_observation, satellites_remaining_capacity, remove = True)
	# add the new observation
	user_solution.scheduled_observations.append((dcop_observation, start_time))
	satellite_name = dcop_observation.satellite.name
	# update the satellite remaining capacity
	update_satellite_capacity([(dcop_observation, start_time)], satellites_remaining_capacity)
	print(f"DCOP for request {request.name} scheduled : {dcop_observation.name} at start time {start_time}.")
	# update non exclusive requests completed
	non_exclusive_requests_completed.append(request.name)
	
# 3) création de l'assignement des observations hors des fenêtres exclusives avec greedy()
non_exclusive_user = problem.users["u_0"]
observations_outside_exclusive_windows = problem.get_observations_outside_exclusive_windows()
requests = set([obs.request for obs in observations_outside_exclusive_windows.values()])
requests = [problem.get_request_copy_without_observations(req) for req in requests]
# filter requests to only those not already completed
requests = [req for req in requests if req.name not in non_exclusive_requests_completed]
observations_outside_exclusive_windows = {obs.name: obs for obs in observations_outside_exclusive_windows.values() if obs.request.name in [req.name for req in requests]}
# add the observations to each request
for req in requests:
	for obs in observations_outside_exclusive_windows.values():
		if obs.request.name == req.name:
			req.observations.append(obs)
# create a temporary problem with these modified requests
non_exclusive_problem = Problem(
	satellites = problem.satellites,
	users = {"u_0": non_exclusive_user},
	requests = {req.name: req for req in requests},
	observations = observations_outside_exclusive_windows
)
non_exclusive_solution = greedy_solver(non_exclusive_problem, satellites_remaining_capacity)
print(f"Non-exclusive observations scheduled: {len(non_exclusive_solution.scheduled_observations)}")
# update satellites_remaining_capacity based on non-exclusive solution
# update_satellite_capacity(non_exclusive_solution.scheduled_observations, satellites_remaining_capacity)

	

# 4) Enfin on combine les assignements trouvés pour avoir la solution final
final_solution = combine_solutions([non_exclusive_solution] + list(exclusive_users_solutions.values()), problem)
is_valid = final_solution.verify()
total_reward = final_solution.total_reward()
print(f"\nFinal solution after DCOP adjustments: valid={is_valid}, total_reward={total_reward}, scheduled_observations={len(final_solution.scheduled_observations)}")
plot_instance(problem, final_solution, display_in_browser=True, title="Final Solution after DCOP Adjustments v1")	# time window of the example

In [ ]:
# VERSION 2 (avec partie 2 et 3 inversée)

# RESOLUTION s_dcop EOSCSP solver
# Steps :
# 0) input : instance du problème
# 1) pour chaque user exclusif :
# 	création de l'assignement des observations de l'utilisateur exclusif avec greedy()
# 3) création de l'assignement des observations hors des fenêtres exclusives avec greedy()	(changé l'ordre des étapes 2 et 3)
# 2) pour chaque requète (non exclusive) restante, dans l'ordre :
# 	création d'un DCOP avec tous les assignements créés jusqu'à présent ainsi que les observations de la requète actuelle
# 	la résolution du DCOP renvoie des nouvelles assignations pour les zones des utilisateurs exclusifs
# 4) Enfin on combine les assignements trouvés pour avoir la solution final

def update_satellite_capacity(scheduled_observations : list[tuple[Observation, int]], satellites_remaining_capacity : dict, remove: bool = False):
	# function to update the satellites_remaining_capacity based on the scheduled_observations
	for scheduled_observation, start_time in scheduled_observations:
		sat_name = scheduled_observation.satellite.name
		if sat_name in satellites_remaining_capacity:
			if remove:
				satellites_remaining_capacity[sat_name] += 1
			else:
				satellites_remaining_capacity[sat_name] -= 1

# problem = random_instance_generator_paper.generate_instance()
problem = random_instance_generator.generate_instance()
# problem = problem_example
# problem = new_example_problem_example

# 0) tracker les capacités des satellites restantes
satellites_remaining_capacity = {}
for sat in problem.satellites.values():
	satellites_remaining_capacity[sat.name] = sat.capacity

# 1) pour chaque user exclusif :
exclusive_users = problem.get_exclusive_users()
# sort exclusive users by priority
exclusive_users = dict(sorted(exclusive_users.items(), key=lambda item: item[1].priority))
exclusive_users_solutions = {}
for user in exclusive_users.values():
	# création de l'assignement des observations de l'utilisateur exclusif avec greedy()
	user_problem = Problem(
		satellites = problem.satellites,
		users = {user.name: user},
		requests = problem.get_user_requests(user),
		observations = problem.get_user_observations(user)
	)
	user_solution = greedy_solver(user_problem, satellites_remaining_capacity)
	# reduce capacity based on user_solution assignments
	# update_satellite_capacity(user_solution.scheduled_observations, satellites_remaining_capacity)
	exclusive_users_solutions[user.name] = user_solution
	print(f"User {user.name} exclusive observations scheduled: {len(user_solution.scheduled_observations)}")

# 3) création de l'assignement des observations hors des fenêtres exclusives avec greedy()
non_exclusive_user = problem.users["u_0"]
observations_outside_exclusive_windows = problem.get_observations_outside_exclusive_windows()
requests = set([obs.request for obs in observations_outside_exclusive_windows.values()])
requests = [problem.get_request_copy_without_observations(req) for req in requests]
for req in requests:
	for obs in observations_outside_exclusive_windows.values():
		if obs.request.name == req.name:
			req.observations.append(obs)
# create a temporary problem with these modified requests
non_exclusive_problem = Problem(
	satellites = problem.satellites,
	users = {"u_0": non_exclusive_user},
	requests = {req.name: req for req in requests},
	observations = observations_outside_exclusive_windows
)
non_exclusive_solution = greedy_solver(non_exclusive_problem, satellites_remaining_capacity)
print(f"Non-exclusive observations scheduled: {len(non_exclusive_solution.scheduled_observations)}")
# update satellites_remaining_capacity based on non-exclusive solution
# update_satellite_capacity(non_exclusive_solution.scheduled_observations, satellites_remaining_capacity)

# 2.0) combinaison des solutions partielles
initial_solution = combine_solutions([non_exclusive_solution] + list(exclusive_users_solutions.values()), problem)
# 2) pour chaque requète non assignée restante (non exclusive) :
non_exclusive_requests = problem.get_non_exclusive_requests()
uncompleted_requests = initial_solution.get_uncompleted_requests()
remaining_requests = {
	req_name: req
	for req_name, req in non_exclusive_requests.items()
	if req_name in uncompleted_requests
}
# sort remaining requests by reward descending
remaining_requests = dict(sorted(remaining_requests.items(), key=lambda item: item[1].reward, reverse=True))
for request in remaining_requests.values():
	# création d'un DCOP avec tous les assignements créés jusqu'à présent ainsi que les observations de la requète actuelle
	# la résolution du DCOP renvoie des nouvelles assignations pour les zones des utilisateurs exclusifs
	dcop, variables_mapping, assigned_observations_start_times = get_DCOP_parameters(request, non_exclusive_solution, exclusive_users_solutions, problem, satellites_remaining_capacity)
	yaml_content = convert_DCOP_to_yaml(dcop, variables_mapping)
	yaml_file_path = write_yaml_file(yaml_content, keep_file=True)
	dcop_return = pydcop_solve(yaml_file_path, "dpop", timeout=10, verbose=False)  # timeout of 10 seconds for each DCOP
	if dcop_return is None:
		print(f"No solution found for DCOP of request {request.name}, skipping to next request.")
		continue
	dcop_return = create_solution_from_dcop_return(dcop_return, variables_mapping, assigned_observations_start_times, problem)
	if dcop_return is None:
		print(f"No observation scheduled in DCOP solution for request {request.name}, skipping to next request.")
		continue
	dcop_observation, start_time = dcop_return
	# update the corresponding exclusive user's solution with the new assignments from the DCOP solution
	exclusive_user_doing_the_observation = get_exclusive_user_from_observation_assignment(dcop_observation)
	user_name = exclusive_user_doing_the_observation.name
	user_solution = exclusive_users_solutions[user_name]
	# remove any observation on the same satellite at overlapping times
	removed_observation = remove_overlapping_observations(dcop_observation, start_time, user_solution)
	update_satellite_capacity(removed_observation, satellites_remaining_capacity, remove = True)
	# add the new observation
	user_solution.scheduled_observations.append((dcop_observation, start_time))
	satellite_name = dcop_observation.satellite.name
	# update the satellite remaining capacity
	update_satellite_capacity([(dcop_observation, start_time)], satellites_remaining_capacity)
	print(f"DCOP for request {request.name} scheduled : {dcop_observation.name} at start time {start_time}.")
	

	

# 4) Enfin on combine les assignements trouvés pour avoir la solution final
final_solution = combine_solutions([non_exclusive_solution] + list(exclusive_users_solutions.values()), problem)
is_valid = final_solution.verify()
total_reward = final_solution.total_reward()
proportion_of_scheduled_requests, n_completed_requests, n_all_requests = final_solution.get_proportion_of_completed_requests()
print(f"\nFinal solution after DCOP adjustments: valid={is_valid}, total_reward={total_reward}, Scheduled requests: {proportion_of_scheduled_requests*100:.2f}% ({n_completed_requests} out of {n_all_requests})")
plot_instance(problem, final_solution, display_in_browser=True, title="Final Solution after DCOP Adjustments v2")	# time window of the example

# analyse ?

# PARTIE 2 - Approches par enchères

In [ ]:
# Auction Helper Functions

def compute_marginal_bid(user: User, request: Request, current_solution: Solution, problem: Problem) -> tuple[int, Observation, int]:
	# Calculates the marginal utility of adding a request to a user's current plan.
	best_gain = -1
	best_observation = None
	best_start_time = None

	# Evaluate all possible observations for request
	for observation in request.observations:
		is_in_exclusive = False
		for window in user.exclusive_windows:
			if window.can_potentially_assign_observation(observation):
				is_in_exclusive = True
				break
		
		if not is_in_exclusive:
			continue

		satellite_schedule = current_solution.get_satellite_schedule(observation.satellite)
		extended_schedule = sort_and_extend_satellite_schedule(observation.satellite, satellite_schedule)
		
		# Calculate max reward gain if we insert this observation
		# PSI: no account for synergy, this handles removing overlapping lower-reward observations
		gain, start_time = get_best_observation_replacement(observation, extended_schedule, problem)
		
		if gain > best_gain:
			best_gain = gain
			best_observation = observation
			best_start_time = start_time
			
	return best_gain, best_observation, best_start_time

## PSI: Parallel Single-item Auction

In [ ]:
def psi_solver(problem: Problem) -> Solution:
    # Implementation of Algorithm 2 from the paper
    
    exclusive_users = problem.get_exclusive_users()
    non_exclusive_requests = problem.get_non_exclusive_requests()
    
    # Global satellite capacity tracker
    satellite_capacities = {sat.name: sat.capacity for sat in problem.satellites.values()}
    # to store local plans and collected bids
    user_solutions = {}
    # all_bids[user_name][request_name] = (bid_value, observation, start_time)
    all_bids = {u_name: {} for u_name in exclusive_users}

	# For each exclusive user (lines 2-4)
    for u_name, user in exclusive_users.items():
        # solve its private requests
        user_prob = Problem(
            satellites=problem.satellites,
            users={u_name: user},
            requests=problem.get_user_requests(user),
            observations=problem.get_user_observations(user)
        )
        user_solution = greedy_solver(user_prob, satellite_capacities.copy())
        user_solutions[u_name] = user_solution
        
        # For each request in central planner, calculate the bid based on the exclusive user's current plan
        for r_name, request in non_exclusive_requests.items():
            bid_value, observation, start = compute_marginal_bid(user, request, user_solution, problem)
            all_bids[u_name][r_name] = (bid_value, observation, start)

    for solution in user_solutions.values():
        update_satellite_capacity(solution.scheduled_observations, satellite_capacities)

    # Allocation Phase (lines 5-8)
    allocated_req_names = []
    for r_name, request in non_exclusive_requests.items():
        # Find winner w: user with highest reward bid
        best_bid = 0
        winner_name = None
        
        for u_name in exclusive_users:
            bid_value, _, _ = all_bids[u_name][r_name]
            if bid_value > best_bid:
                best_bid = bid_value
                winner_name = u_name
        
        # Update winner's internal plan
        if winner_name:
            bid_value, winning_obs, winning_start = all_bids[winner_name][r_name]
            
            if satellite_capacities[winning_obs.satellite.name] > 0:
                user_solution = user_solutions[winner_name]
                
				# PSI: no synergy: removing overlap by keeping only the observation with the highest reward
                removed = remove_overlapping_observations(winning_obs, winning_start, user_solution)
                update_satellite_capacity(removed, satellite_capacities, remove=True)
                
                user_solution.scheduled_observations.append((winning_obs, winning_start))
                update_satellite_capacity([(winning_obs, winning_start)], satellite_capacities)
                
                allocated_req_names.append(r_name)

    # Central planner tries to schedule remaining requests outside exclusive portions (line 9)
    remaining_requests = [req for name, req in non_exclusive_requests.items() if name not in allocated_req_names]
    
    observations_outside_exclusive_windows = problem.get_observations_outside_exclusive_windows()
    observations_outside_exclusive_windows_filtered = {o_n: o for o_n, o in observations_outside_exclusive_windows.items() if o.request.name in [r.name for r in remaining_requests]}
    
    u0_prob = Problem(
        satellites=problem.satellites,
        users={"u_0": problem.users["u_0"]},
        requests={r.name: r for r in remaining_requests},
        observations=observations_outside_exclusive_windows_filtered
    )
    u0_solution = greedy_solver(u0_prob, satellite_capacities)

    final_sol = combine_solutions(list(user_solutions.values()) + [u0_solution], problem)
    return final_sol

In [ ]:
problem = random_instance_generator.generate_instance()

start_time = time.time()
solution_psi = psi_solver(problem)
print(f"PSI Solver Time: {(time.time() - start_time)*1000:.2f} ms")

proportion_of_scheduled_requests_psi, n_completed_requests_psi, n_all_requests_psi = solution_psi.get_proportion_of_completed_requests()
print(f"\nPSI Solution: valid={solution_psi.verify()}, total_reward={solution_psi.total_reward()}, Scheduled requests: {proportion_of_scheduled_requests_psi*100:.2f}% ({n_completed_requests_psi} out of {n_all_requests_psi})")
# plot_instance(problem, solution_psi, display_in_browser=True, title="PSI Solution")

## SSI: Sequential Single-item Auction

In [ ]:
def ssi_solver(problem: Problem) -> Solution:
	# Implementation of Algorithm 3 from the paper
	
	exclusive_users = problem.get_exclusive_users()
	non_exclusive_requests_dict = problem.get_non_exclusive_requests()
	
	# Global satellite capacity tracker
	satellite_capacities = {sat.name: sat.capacity for sat in problem.satellites.values()}
	user_solutions = {}

	# For each exclusive user (line 2))
	for u_name, user in exclusive_users.items():
		# solve its private requests
		user_prob = Problem(
			satellites=problem.satellites,
			users={u_name: user},
			requests=problem.get_user_requests(user),
			observations=problem.get_user_observations(user)
		)
		user_solution = greedy_solver(user_prob, satellite_capacities.copy())
		user_solutions[u_name] = user_solution
		update_satellite_capacity(user_solution.scheduled_observations, satellite_capacities)

	# Sorting Requests (line 3)
	non_exclusive_requests = list(non_exclusive_requests_dict.values())
	non_exclusive_requests.sort(key=lambda r: r.reward, reverse=True)

	allocated_req_names = []

	# Sequential rounds (lines 3-8)
	for request in non_exclusive_requests:
		best_bid = 0
		winner_name = None
		winning_obs = None
		winning_start = None
		
		for u_name, user in exclusive_users.items():
			bid_value, observation, start_time = compute_marginal_bid(user, request, user_solutions[u_name], problem)
			if bid_value > best_bid:
				best_bid = bid_value
				winner_name = u_name
				winning_obs = observation
				winning_start = start_time
		
		# Update winner's internal plan
		if winner_name:
			
			if satellite_capacities[winning_obs.satellite.name] > 0:
				user_solution = user_solutions[winner_name]
				
				removed = remove_overlapping_observations(winning_obs, winning_start, user_solution)
				update_satellite_capacity(removed, satellite_capacities, remove=True)
				
				user_solution.scheduled_observations.append((winning_obs, winning_start))
				update_satellite_capacity([(winning_obs, winning_start)], satellite_capacities)
				
				allocated_req_names.append(request.name)

	# Central planner tries to schedule remaining requests outside exclusive portions (line 9)
	remaining_requests = [req for req in non_exclusive_requests if req.name not in allocated_req_names]
	
	observations_outside_exclusive_windows = problem.get_observations_outside_exclusive_windows()
	observations_outside_exclusive_windows_filtered = {o_n: o for o_n, o in observations_outside_exclusive_windows.items() if o.request.name in [r.name for r in remaining_requests]}
	
	u0_prob = Problem(
		satellites=problem.satellites,
		users={"u_0": problem.users["u_0"]},
		requests={r.name: r for r in remaining_requests},
		observations=observations_outside_exclusive_windows_filtered
	)
	u0_solution = greedy_solver(u0_prob, satellite_capacities)

	final_sol = combine_solutions(list(user_solutions.values()) + [u0_solution], problem)
	return final_sol

In [ ]:
problem = random_instance_generator.generate_instance()

start_time = time.time()
solution_ssi = ssi_solver(problem)
print(f"SSI Solver Time: {(time.time() - start_time)*1000:.2f} ms")

proportion_of_scheduled_requests_ssi, n_completed_requests_ssi, n_all_requests_ssi = solution_ssi.get_proportion_of_completed_requests()
print(f"\nSSI Solution: valid={solution_ssi.verify()}, total_reward={solution_ssi.total_reward()}, Scheduled requests: {proportion_of_scheduled_requests_ssi*100:.2f}% ({n_completed_requests_ssi} out of {n_all_requests_ssi})")
# plot_instance(problem, solution_ssi, display_in_browser=True, title="SSI Solution")

In [ ]:
# PSI vs SSI
print("\nComparison PSI vs SSI:")
psi_percentages, ssi_percentages = [], []
psi_times, ssi_times = [], []
for i in range(50):
	problem = random_instance_generator.generate_instance()
	start_time = time.time()
	solution_psi = psi_solver(problem)
	psi_time = (time.time() - start_time)*1000
	proportion_of_scheduled_requests_psi, _, _ = solution_psi.get_proportion_of_completed_requests()
	psi_percentages.append(proportion_of_scheduled_requests_psi*100)
	psi_times.append(psi_time)
	start_time = time.time()
	solution_ssi = ssi_solver(problem)
	ssi_time = (time.time() - start_time)*1000
	proportion_of_scheduled_requests_ssi, _, _ = solution_ssi.get_proportion_of_completed_requests()
	ssi_percentages.append(proportion_of_scheduled_requests_ssi*100)
	ssi_times.append(ssi_time)
for i in range(50): # the other way around just in case
	problem = random_instance_generator.generate_instance()
	start_time = time.time()
	solution_ssi = ssi_solver(problem)
	ssi_time = (time.time() - start_time)*1000
	proportion_of_scheduled_requests_ssi, _, _ = solution_ssi.get_proportion_of_completed_requests()
	ssi_percentages.append(proportion_of_scheduled_requests_ssi*100)
	ssi_times.append(ssi_time)
	start_time = time.time()
	solution_psi = psi_solver(problem)
	psi_time = (time.time() - start_time)*1000
	proportion_of_scheduled_requests_psi, _, _ = solution_psi.get_proportion_of_completed_requests()
	psi_percentages.append(proportion_of_scheduled_requests_psi*100)
	psi_times.append(psi_time)
average_psi_percentage = sum(psi_percentages) / len(psi_percentages)
average_ssi_percentage = sum(ssi_percentages) / len(ssi_percentages)
average_psi_time = sum(psi_times) / len(psi_times)
average_ssi_time = sum(ssi_times) / len(ssi_times)
print(f"Average PSI Scheduled Requests: {average_psi_percentage:.2f}%, Average Time: {average_psi_time:.2f} ms")
print(f"Average SSI Scheduled Requests: {average_ssi_percentage:.2f}%, Average Time: {average_ssi_time:.2f} ms")

#### Results
SSI is slightly better and slower, which is expected

## Regret-based SSI